In [10]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
import pickle

# -----------------------------
# Helper: Extract soil type
# -----------------------------
def extract_soil(text):
    text = str(text).lower()
    if "loamy" in text:
        return "loamy"
    elif "clay" in text:
        return "clayey"
    elif "sandy" in text:
        return "sandy"
    else:
        return "unknown"

# -----------------------------
# 1️⃣ Trees
# -----------------------------
trees_df = pd.read_csv("datasets/Commercial_trees_UttarPradesh.csv", header=None)
trees_df.columns = ['District', 'Tree', 'Category', 'Requirements']
trees_df = trees_df[trees_df['Category'] == "Soil, Water & pH"].copy()
trees_df['Tree'] = trees_df['Tree'].str.split("—").str[0].str.strip()
trees_df['Soil'] = trees_df['Requirements'].apply(extract_soil)
trees_df['Type'] = 'Tree'
trees_df = trees_df[['District','Tree','Soil','Type']].rename(columns={'Tree':'Plant'})

# -----------------------------
# 2️⃣ Crops
# -----------------------------
crops_df = pd.read_csv("datasets/Field_Crop_UttarPradesh.csv")
crops_df.columns = ['District','Crop','Category','Requirements']
crops_df = crops_df[crops_df['Category'] == "Soil, Water & pH"].copy()
crops_df['Crop'] = crops_df['Crop'].str.strip()
crops_df['Soil'] = crops_df['Requirements'].apply(extract_soil)
crops_df['Type'] = 'Crop'
crops_df = crops_df[['District','Crop','Soil','Type']].rename(columns={'Crop':'Plant'})

# -----------------------------
# 3️⃣ Flowers
# -----------------------------
flowers_df = pd.read_csv("datasets/flori.csv")
flowers_df = flowers_df.rename(columns={
    'Major Commercial Flowers': 'Flower',
    'Soil and Water Requirement': 'Requirements'
})
flowers_df['Flower'] = flowers_df['Flower'].str.split(',')
flowers_df = flowers_df.explode('Flower')
flowers_df['Flower'] = flowers_df['Flower'].str.strip()
flowers_df['Soil'] = flowers_df['Requirements'].apply(extract_soil)
flowers_df['Type'] = 'Flower'
flowers_df = flowers_df[['District','Flower','Soil','Type']].rename(columns={'Flower':'Plant'})

# -----------------------------
# 4️⃣ Combine datasets
# -----------------------------
combined_df = pd.concat([trees_df, crops_df, flowers_df], ignore_index=True)

# -----------------------------
# 5️⃣ Pivot for multi-output
# -----------------------------
train_df = combined_df.pivot_table(
    index=['District','Soil'],
    columns='Type',
    values='Plant',
    aggfunc=lambda x: x.iloc[0]
).reset_index().fillna("Unknown")

# -----------------------------
# 6️⃣ Fill flowers by district if Unknown
# -----------------------------
flower_map = flowers_df.groupby('District')['Plant'].apply(list).to_dict()

def assign_flower(row):
    if row['Flower'] == "Unknown":
        district = row['District']
        return flower_map.get(district, ["Unknown"])[0]  # pick first flower
    return row['Flower']

train_df['Flower'] = train_df.apply(assign_flower, axis=1)

# -----------------------------
# 7️⃣ Encode
# -----------------------------
district_encoder = LabelEncoder()
soil_encoder = LabelEncoder()
tree_encoder = LabelEncoder()
crop_encoder = LabelEncoder()
flower_encoder = LabelEncoder()

train_df['district_enc'] = district_encoder.fit_transform(train_df['District'])
train_df['soil_enc'] = soil_encoder.fit_transform(train_df['Soil'])
train_df['tree_enc'] = tree_encoder.fit_transform(train_df['Tree'])
train_df['crop_enc'] = crop_encoder.fit_transform(train_df['Crop'])
train_df['flower_enc'] = flower_encoder.fit_transform(train_df['Flower'])

X = train_df[['district_enc','soil_enc']]
y = train_df[['tree_enc','crop_enc','flower_enc']]

# -----------------------------
# 8️⃣ Train MultiOutput model
# -----------------------------
model = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
model.fit(X, y)

# -----------------------------
# 9️⃣ Save model and encoders
# -----------------------------
pickle.dump(model, open("IFS_model.pkl","wb"))
pickle.dump(district_encoder, open("district_encoder.pkl","wb"))
pickle.dump(soil_encoder, open("soil_encoder.pkl","wb"))
pickle.dump(tree_encoder, open("tree_encoder.pkl","wb"))
pickle.dump(crop_encoder, open("crop_encoder.pkl","wb"))
pickle.dump(flower_encoder, open("flower_encoder.pkl","wb"))

print("✅ Model and encoders saved successfully!")


✅ Model and encoders saved successfully!


In [11]:
from flask import Flask, render_template, request
import pickle
import requests
import numpy as np

app = Flask(__name__)

# Load model and encoders
model = pickle.load(open("IFS_model.pkl","rb"))
district_encoder = pickle.load(open("district_encoder.pkl","rb"))
soil_encoder = pickle.load(open("soil_encoder.pkl","rb"))
tree_encoder = pickle.load(open("tree_encoder.pkl","rb"))
crop_encoder = pickle.load(open("crop_encoder.pkl","rb"))
flower_encoder = pickle.load(open("flower_encoder.pkl","rb"))

# -----------------------------
# Weather
# -----------------------------
def get_weather(district):
    try:
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={district}&count=1&language=en&format=json"
        geo_data = requests.get(geo_url).json()
        if "results" not in geo_data or len(geo_data["results"]) == 0:
            return {"temp": "N/A", "description": "Not Found"}
        lat = geo_data["results"][0]["latitude"]
        lon = geo_data["results"][0]["longitude"]
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        data = requests.get(weather_url).json()
        temp = data["current_weather"]["temperature"]
        code = data["current_weather"]["weathercode"]
        return {"temp": temp, "description": f"Weather Code: {code}"}
    except:
        return {"temp":"N/A","description":"Not Found"}

# -----------------------------
# Predict recommendation
# -----------------------------
@app.route('/predict', methods=['POST'])
def predict():
    district = request.form['district']
    soil = request.form.get('soil','loamy')
    lang = request.form.get('language','en')
    land_area = 1.0

    # Encode inputs
    district_enc = district_encoder.transform([district])[0]
    soil_enc = soil_encoder.transform([soil])[0]

    pred_enc = model.predict(np.array([[district_enc, soil_enc]]))[0]

    rec = {
        "tree": tree_encoder.inverse_transform([pred_enc[0]])[0],
        "crop": crop_encoder.inverse_transform([pred_enc[1]])[0],
        "flower": flower_encoder.inverse_transform([pred_enc[2]])[0]
    }

    # Language translations
    if lang=='hi':
        translation = {"tree":"पेड़","crop":"फसल","flower":"फूल","recommended":"एकीकृत खेती प्रणाली की अनुशंसा"}
    else:
        translation = {"tree":"Tree","crop":"Crop","flower":"Flower","recommended":"Integrated Farming System Recommendation"}

    weather = get_weather(district)

    return render_template("result.html",
                           district=district,
                           soil=soil,
                           land_area=land_area,
                           rec=rec,
                           translation=translation,
                           weather=weather,
                           lang=lang)
